# E2: finite native rigid-ligand collection receiving

Reproduce the admitted P59/P69 source recipes, named Gasteiger–Marsili charges and
AutoDock4 chemical-environment types. Both exact prepared PDBQT byte gates pass
before importing Vina. Inspection JSON is comparison evidence, never MM restoration.
No H generation, additional molecular matcher, retry, replacement or population expansion.

One public serial `dock_many` selection: P59 / unsupported constraints / P69.
Pinned Vina1.2.7, CPU1, seed42 for both, exhaustiveness1, max3 poses, range3 kcal/mol,
common30×24×24 Å joint box and external protein-only receptor. All ordinary outcomes
are saved before qualification; source/memory/process-control errors propagate with
partial records retained. Native searches perform their own internal calculations.

Direct caps: conversions6, Q2, typing2, prepare2, gate exports2, dock_many1,
outcome restores3, result audits2. Backend exports2 observed separately; native
constructor/receptor/ligand-string/maps/dock/energies2, poses2coords+2text, info4.
Native score/optimize/randomize/set_weights/file-ligand0. No extra engine capture call.
Numeric returns are retained first as unpickled NPY arrays with unit/shape/dtype/hash,
then as finite JSON. Provider internals and public snapshot helpers are observed
separately. No new source readiness/CIP/self-witness requests.

The embedded caller-owned pure report preserves five native columns and original
rank-1 representatives, exact ties and unranked reasons. It never modifies source
ranks/histories, calls E1 same-state comparison, or claims affinity, convergence,
stereoselectivity or screening effectiveness. The joint domain covers different sites.

Only the registered isolated worker/guard executes this unexecuted notebook:
CPU0/nice10/one worker, reserve8GiB, owned RSS1GiB, AS2GiB, CPU60s/wall90s,
owned TMPDIR. Env DOCKINGMT_E2_ROOT/OUTPUT and DOCKINGMT_E2_ prefixes for
CANDIDATE_MANIFEST/_SHA256, SOURCE_ARCHIVE/_SHA256, PREPARATION_ARCHIVE/_SHA256,
STUDY_POLICY/_SHA256. Worker injects registered_installed_check and qualifies
runtime/private payload/engine origins. Original and restored outcomes are both kept;
coordinate roundtrips compare physical Å, not representation bytes.


In [ ]:
import base64
import gzip
import hashlib
import inspect
import json
import math
import os
import sys
import tempfile
import weakref
from dataclasses import asdict
from datetime import datetime, timezone
from functools import wraps
from pathlib import Path, PurePosixPath

root = Path(os.environ['DOCKINGMT_E2_ROOT']).resolve()
output = Path(os.environ['DOCKINGMT_E2_OUTPUT']).resolve()
assert not output.exists() and not root.is_relative_to(output)
output.mkdir(parents=True, exist_ok=False)
manifest_path = Path(os.environ['DOCKINGMT_E2_CANDIDATE_MANIFEST']).resolve()
manifest_sha = os.environ['DOCKINGMT_E2_CANDIDATE_MANIFEST_SHA256']
expected_manifest_sha = (
    '898b26fa60561bb99f823d2dd0c27216cc5771ac7dceacdb103760fed2e22284'
)
expected_candidates = [
    dict(
        ligand_id='p59',
        state_id='e2-5x72-p59',
        path='tests/data/vina_torsions/5x72_ligand_p59H.sdf',
        sha256='ef26c05a198a16972efcf8baae644f6b38fd953561f130ee230626088766b08a',
        expected_cip='R',
        source_stereo_atom_index=7,
    ),
    dict(
        ligand_id='p69',
        state_id='e2-5x72-p69',
        path='tests/data/vina_torsions/5x72_ligand_p69H.sdf',
        sha256='637a992134b2038a0ea3cff0c1f8355ea232720c9eae5347100b7186170fcdf1',
        expected_cip='S',
        source_stereo_atom_index=7,
    ),
]
budgets = dict(
    sdf_native=2,
    native_to_rdkit=2,
    rdkit_to_native=2,
    charge_assignment=2,
    type_assignment=2,
    prepare=2,
    to_pdbqt=2,
    dock_many=1,
    audit_result=2,
    public_outcome_restores=3,
)
counts = dict.fromkeys(budgets, 0)
observed = dict(
    public_convert_alias=0,
    public_get_alias=0,
    snapshot=0,
    internal_merge=0,
    writer_exports=0,
    known_source_readers=0,
)
native_caps = dict(
    constructor=2,
    set_receptor=2,
    set_ligand_from_string=2,
    set_ligand_from_file=0,
    compute_vina_maps=2,
    dock=2,
    energies=2,
    poses_coordinates=2,
    poses_text=2,
    info=4,
    score=0,
    optimize=0,
    randomize=0,
    set_weights=0,
)
native_counts = dict.fromkeys(native_caps, 0)
item_specs = [
    dict(position=0, run_id='p59-main', ligand_id='p59', role='scientific_candidate'),
    dict(
        position=1,
        run_id='p59-unsupported-intent-control',
        ligand_id='p59',
        role='unsupported_constraints_control',
    ),
    dict(position=2, run_id='p69-main', ligand_id='p69', role='scientific_candidate'),
]
item_ledger = [dict(**row, status='not_attempted') for row in item_specs]
expected_pdbqt = dict(
    p59='ae0e9d3beb4d14363ea6edb1bff591c56d53b02295082bae8c281d9284d84b9e',
    p69='1cbe49d31e1859e34b7cc71af1a8514fef9314401401b2b76b6a7e07dfc7ebaa',
)
forbidden_attempts = []
ledger = [
    dict(
        position=i + 1,
        ligand_id=c['ligand_id'],
        candidate_state_id=c['state_id'],
        status='not_attempted',
        stages={},
    )
    for i, c in enumerate(expected_candidates)
]
current = None
stage = 'custody'


def file_sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def save_json(path, value):
    payload = json.dumps(value, indent=2, allow_nan=False) + '\n'
    with path.open('x') as stream:
        stream.write(payload)


def persist_ledger():
    (output / 'ledger.json').write_text(
        json.dumps(
            dict(preparations=ledger, items=item_ledger), indent=2, allow_nan=False
        )
        + '\n'
    )


def event(operation, **fields):
    with (output / 'operations.jsonl').open('a') as stream:
        stream.write(
            json.dumps(
                dict(
                    operation=operation,
                    stage=stage,
                    candidate=None if current is None else current['ligand_id'],
                    at_utc=datetime.now(timezone.utc).isoformat(),
                    direct_counts=counts.copy(),
                    native_counts=native_counts.copy(),
                    **fields,
                ),
                allow_nan=False,
            )
            + '\n'
        )


persist_ledger()
try:
    raw_manifest = manifest_path.read_bytes()
    (output / 'candidate_manifest.json').write_bytes(raw_manifest)
    assert (
        manifest_sha
        == expected_manifest_sha
        == hashlib.sha256(raw_manifest).hexdigest()
    )
    candidate_manifest = json.loads(raw_manifest)
    assert candidate_manifest['schema'] == 'dockingmt.e2_ligand_source_candidates@1'
    assert candidate_manifest['owner'] == 'uibcdf/dockingmt#81'
    assert len(candidate_manifest['candidates']) == 2
    assert candidate_manifest['source_license']['spdx'] == 'Apache-2.0'
    assert (
        candidate_manifest['source_license']['commit']
        == '3c65c0b3e6c2c1d183f6a175ecb65e3c5ba91645'
    )
    for actual, expected in zip(candidate_manifest['candidates'], expected_candidates):
        assert {key: actual[key] for key in expected} == expected
    stored_inputs = {str(manifest_path): manifest_sha}
    input_copies = {}
    inputs = output / 'inputs'
    inputs.mkdir()
    assert len(candidate_manifest['files']) == 7
    seen = set()
    for row in candidate_manifest['files']:
        relative = PurePosixPath(row['path'])
        assert not relative.is_absolute() and '..' not in relative.parts
        assert str(relative) == row['path'] and row['path'] not in seen
        seen.add(row['path'])
        path = root / relative
        assert path.resolve().is_relative_to(root)
        data = path.read_bytes()
        copy_path = inputs / relative.name
        copy_path.write_bytes(data)
        assert (
            len(data) == row['bytes']
            and hashlib.sha256(data).hexdigest() == row['sha256']
        )
        stored_inputs[str(path)] = row['sha256']
        input_copies[str(copy_path)] = row['sha256']
    for row in expected_candidates:
        assert stored_inputs[str(root / row['path'])] == row['sha256']
    assert os.sched_getaffinity(0) == {0} and os.getpriority(os.PRIO_PROCESS, 0) >= 10
    meminfo = dict(
        line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines()
    )
    assert int(meminfo['MemAvailable'].split()[0]) * 1024 >= 8 * 1024**3
    assert not any(name.split('.')[0] == 'vina' for name in sys.modules)
except BaseException as error:
    save_json(
        output / 'receipt.json',
        dict(
            schema='dockingmt.e2_ligand_collection_receiving@1',
            owner='uibcdf/dockingmt#81',
            outcome='FAIL',
            stage=stage,
            error_type=type(error).__name__,
            reason=str(error),
            direct_counts=counts,
            denominator=2,
            ledger=ledger,
            retry_permitted=False,
        ),
    )
    raise
# Source records are comparisons only, never molecular restoration inputs.
archive = Path(os.environ['DOCKINGMT_E2_SOURCE_ARCHIVE']).resolve()
archive_sha = os.environ['DOCKINGMT_E2_SOURCE_ARCHIVE_SHA256']
try:
    archive_manifest_path = archive / 'manifest.json'
    assert (
        archive_sha
        == 'a5a60b65715b466bd04dc5e2bdb41a73489fdf9a13254552870628d387221b8d'
    )
    assert file_sha(archive_manifest_path) == archive_sha
    archive_manifest = json.loads(archive_manifest_path.read_text())
    assert archive_manifest['owner'] == 'uibcdf/dockingmt#81'
    assert (
        archive_manifest['molecular_receiving_outcome']
        == archive_manifest['persistence_outcome']
        == 'PASS'
    )
    stored_inputs[str(archive_manifest_path)] = archive_sha
    (output / 'source_archive_manifest.json').write_bytes(
        archive_manifest_path.read_bytes()
    )
    for row in archive_manifest['files']:
        relative = PurePosixPath(row['path'])
        assert not relative.is_absolute() and '..' not in relative.parts
        path = archive / relative
        assert path.resolve().is_relative_to(archive)
        data = path.read_bytes()
        assert (
            len(data) == row['bytes']
            and hashlib.sha256(data).hexdigest() == row['sha256']
        )
        stored_inputs[str(path)] = row['sha256']
        if str(relative).endswith('.gz'):
            decoded = gzip.decompress(data)
            assert (
                len(decoded) == row['decoded_bytes']
                and hashlib.sha256(decoded).hexdigest() == row['decoded_sha256']
            )
except BaseException as error:
    save_json(
        output / 'receipt.json',
        dict(
            schema='dockingmt.e2_ligand_collection_receiving@1',
            owner='uibcdf/dockingmt#81',
            outcome='FAIL',
            stage='archive-custody',
            reason=str(error),
            direct_counts=counts,
            ledger=ledger,
            retry_permitted=False,
        ),
    )
    raise


def source_record(ligand, filename):
    original = f'attempts/source-01/output/{ligand}/{filename}'
    rows = [
        r
        for r in archive_manifest['files']
        if r.get('original_path', r['path']) == original
    ]
    assert len(rows) == 1, original
    row = rows[0]
    data = (archive / row['path']).read_bytes()
    assert hashlib.sha256(data).hexdigest() == row['sha256']
    return json.loads(gzip.decompress(data) if row['path'].endswith('.gz') else data)


# All historical records remain observations, never molecular restoration inputs.
preparation_archive = Path(os.environ['DOCKINGMT_E2_PREPARATION_ARCHIVE']).resolve()
preparation_archive_sha = os.environ['DOCKINGMT_E2_PREPARATION_ARCHIVE_SHA256']
try:
    path = preparation_archive / 'manifest.json'
    assert (
        preparation_archive_sha
        == 'bd4f34c7c696efa1c864c9ea410630398b5ec164726eeaaf6462ea9d9b78d326'
    )
    assert file_sha(path) == preparation_archive_sha
    preparation_manifest = json.loads(path.read_text())
    assert preparation_manifest['owner'] == 'uibcdf/dockingmt#81'
    assert (
        preparation_manifest['molecular_receiving_outcome']
        == preparation_manifest['persistence_outcome']
        == 'PASS'
    )
    stored_inputs[str(path)] = preparation_archive_sha
    (output / 'preparation_archive_manifest.json').write_bytes(path.read_bytes())
    seen = set()
    for row in preparation_manifest['files']:
        relative = PurePosixPath(row['path'])
        assert not relative.is_absolute() and '..' not in relative.parts
        assert str(relative) == row['path'] and row['path'] not in seen
        seen.add(row['path'])
        path = preparation_archive / relative
        assert path.resolve().is_relative_to(preparation_archive)
        data = path.read_bytes()
        assert (
            len(data) == row['bytes']
            and hashlib.sha256(data).hexdigest() == row['sha256']
        )
        stored_inputs[str(path)] = row['sha256']
        if row['path'].endswith('.gz'):
            decoded = gzip.decompress(data)
            assert len(decoded) == row['decoded_bytes']
            assert hashlib.sha256(decoded).hexdigest() == row['decoded_sha256']
    policy_path = Path(os.environ['DOCKINGMT_E2_STUDY_POLICY']).resolve()
    policy_sha = os.environ['DOCKINGMT_E2_STUDY_POLICY_SHA256']
    assert (
        policy_sha == 'd7bae710203402d37fb1a0378ea2d5c14c1d427bde5575f378cb25552648a34e'
    )
    assert file_sha(policy_path) == policy_sha
    policy = json.loads(policy_path.read_text())
    policy_before = json.loads(json.dumps(policy, allow_nan=False))
    (output / 'study_policy.json').write_bytes(policy_path.read_bytes())
    stored_inputs[str(policy_path)] = policy_sha
    tmpdir = Path(os.environ['TMPDIR']).resolve()
    assert tmpdir.is_dir() and tmpdir.is_relative_to(output.parent)
    assert Path(tempfile.gettempdir()).resolve() == tmpdir
except BaseException as error:
    save_json(
        output / 'receipt.json',
        dict(
            schema='dockingmt.e2_ligand_collection_receiving@1',
            owner='uibcdf/dockingmt#81',
            outcome='FAIL',
            stage='preparation-and-policy-custody',
            error_type=type(error).__name__,
            reason=str(error),
            direct_counts=counts,
            native_counts=native_counts,
            ledger=dict(preparations=ledger, items=item_ledger),
            retry_permitted=False,
        ),
    )
    raise


def prep_record(ligand, filename):
    original = f'attempts/prep-01/output/{ligand}/{filename}'
    rows = [
        row
        for row in preparation_manifest['files']
        if row.get('original_path', row['path']) == original
    ]
    assert len(rows) == 1, original
    row = rows[0]
    data = (preparation_archive / row['path']).read_bytes()
    assert hashlib.sha256(data).hexdigest() == row['sha256']
    return json.loads(gzip.decompress(data) if row['path'].endswith('.gz') else data)

In [ ]:
class RejectVinaImports:
    def find_spec(self, fullname, path=None, target=None):
        if fullname.split('.')[0] == 'vina':
            forbidden_attempts.append('import:' + fullname)
            raise AssertionError(
                'Vina import is forbidden until both exact prepared input gates pass'
            )
        return None


import_guard = RejectVinaImports()
sys.meta_path.insert(0, import_guard)
try:
    import importlib.metadata as metadata

    import molsysmt as msm
    import numpy as np
    import pyunitwizard as puw

    import dockingmt as dmt
    from dmt_test_support.qualify_chemical_templates import detached_record, snapshot
except BaseException as error:
    save_json(
        output / 'receipt.json',
        dict(
            schema='dockingmt.e2_ligand_collection_receiving@1',
            owner='uibcdf/dockingmt#81',
            outcome='FAIL',
            stage='imports',
            error_type=type(error).__name__,
            reason=str(error),
            direct_counts=counts,
            denominator=2,
            ledger=ledger,
            forbidden_attempts=forbidden_attempts,
            retry_permitted=False,
        ),
    )
    raise

patches = []
records = []
prepared_by_id = {}
outcomes = []
outcome_records = []
live_sources = []
identities = {}
installed_check = globals()['registered_installed_check']
caller_policy = detached_record(puw.configure.report())
integrity = dict(before=False, after=False)


def save(path, value):
    save_json(path, detached_record(value))


def patch(owner, name, replacement):
    original = getattr(owner, name)
    patches.append((owner, name, original))
    setattr(owner, name, replacement)


def watch(function, counter):
    @wraps(function)
    def observed_call(*args, **kwargs):
        observed[counter] += 1
        return function(*args, **kwargs)

    return observed_call


def reject(name):
    def forbidden(*args, **kwargs):
        forbidden_attempts.append(name)
        event('forbidden', operation_name=name)
        raise AssertionError('Forbidden collection operation: ' + name)

    return forbidden


def direct(label, budget, function, *args, **kwargs):
    global stage
    stage = label
    assert counts[budget] < budgets[budget], budget
    counts[budget] += 1
    current['stages'][label] = 'started'
    persist_ledger()
    event('started', direct_operation=budget)
    value = function(*args, **kwargs)
    current['stages'][label] = 'returned'
    persist_ledger()
    event('returned', direct_operation=budget)
    return value


def source_identity():
    for name, module in (
        ('dockingmt', dmt),
        ('molsysmt', msm),
        ('numpy', np),
        ('pyunitwizard', puw),
    ):
        path = Path(module.__file__).resolve()
        identities[name] = dict(
            version=metadata.version(name), path=str(path), sha256=file_sha(path)
        )
    for name, function in (
        ('convert', msm.convert),
        ('get', msm.get),
        ('charge_assignment', msm.build.assign_partial_charges),
        ('type_assignment', msm.build.assign_autodock_atom_types),
        ('prepare_ligand', dmt.prepare_ligand),
        ('dock_many', dmt.dock_many),
        ('known_source_reader', msm.form.string_pdbqt_text.read_poses),
        ('snapshot', snapshot),
        ('detached_record', detached_record),
    ):
        function = inspect.unwrap(function)
        path = Path(function.__code__.co_filename).resolve()
        identities[name] = dict(
            module=function.__module__, path=str(path), sha256=file_sha(path)
        )
    helper_sha = file_sha(root / 'dmt_test_support/qualify_chemical_templates.py')
    assert (
        identities['snapshot']['sha256']
        == identities['detached_record']['sha256']
        == helper_sha
    )
    save(output / 'initial_module_identities.json', identities)


export_views = []
live_preparations = []


def inspect_native(system):
    observed['snapshot'] += 1
    result = dict(
        form=msm.get_form(system),
        chemistry_structures=snapshot(system),
        n_atoms=int(msm.get(system, n_atoms=True)),
        n_bonds=int(msm.get(system, n_bonds=True)),
        ids=[str(x) for x in msm.get(system, element='atom', atom_id=True)],
        elements=[
            str(x)
            for x in msm.get(
                system,
                element='atom',
                atom_type=True,
                structure_indices=0,
                chemical_state='structure',
            )
        ],
        coordinates_angstrom=puw.get_value(
            msm.get(system, coordinates=True), to_unit='angstrom'
        ),
        coordinate_unit='angstrom',
        has_partial_charge=bool(msm.has_attribute(system, 'partial_charge')),
        has_atom_ff_type=bool(msm.has_attribute(system, 'atom_ff_type')),
    )
    mechanics = system.molecular_mechanics
    q = (
        msm.get(system, element='atom', partial_charge=True)
        if result['has_partial_charge']
        else None
    )
    if puw.is_quantity(q):
        q = puw.get_value(q, to_unit='elementary_charge')
    result.update(
        mechanics=mechanics.to_dict(),
        charges_e=q,
        charge_unit='elementary_charge',
        types=list(msm.get(system, element='atom', atom_ff_type=True))
        if result['has_atom_ff_type']
        else None,
        partial_charge_assignment=mechanics.partial_charge_assignment,
        atom_type_assignment=mechanics.atom_type_assignment,
    )
    return detached_record(result)


def same_chemistry_geometry(left, right):
    a = detached_record(left['chemistry_structures'])
    b = detached_record(right['chemistry_structures'])
    assert a['structure_units'] == b['structure_units']
    ax = a['structures'].pop('coordinates')
    bx = b['structures'].pop('coordinates')
    np.testing.assert_allclose(ax, bx, rtol=0, atol=1e-13)
    assert a == b
    for field in ('n_atoms', 'n_bonds', 'ids', 'elements', 'coordinate_unit'):
        assert left[field] == right[field], field
    np.testing.assert_allclose(
        left['coordinates_angstrom'], right['coordinates_angstrom'], rtol=0, atol=1e-12
    )


def compare_source_observation(actual, expected):
    same_chemistry_geometry(actual, expected)
    for field in ('form', 'has_partial_charge', 'has_atom_ff_type'):
        assert actual[field] == expected[field], field


def compare_inspection(actual, expected):
    same_chemistry_geometry(actual, expected)
    for name in (
        'form',
        'has_partial_charge',
        'has_atom_ff_type',
        'types',
        'partial_charge_assignment',
        'atom_type_assignment',
    ):
        assert actual[name] == expected[name], name
    if actual['charges_e'] is not None:
        np.testing.assert_allclose(
            actual['charges_e'], expected['charges_e'], rtol=0, atol=1e-12
        )
    assert actual['mechanics'] == expected['mechanics']


writer_records = []
reader_records = []
native_records = []
native_instances = weakref.WeakKeyDictionary()
receptor_paths = []
active_item = None


def install_monitoring():
    original_convert = msm.convert

    @wraps(original_convert)
    def convert(*args, **kwargs):
        observed['public_convert_alias'] += 1
        target = kwargs.get('to_form', 'molsysmt.MolSys')
        if isinstance(target, str) and 'h5msm' in target.lower():
            return reject('H5 persistence')()
        writer = target == 'string:pdbqt_text' and kwargs.get('return_atom_map')
        if writer:
            assert observed['writer_exports'] < 4
            observed['writer_exports'] += 1
        value = original_convert(*args, **kwargs)
        if writer:
            text, mapping = value
            payload = text.removeprefix('pdbqt_text:')
            directory = (
                output / 'writer_exports' / f'export_{observed["writer_exports"]}'
            )
            directory.mkdir(parents=True)
            (directory / 'payload.pdbqt').write_text(payload)
            save(directory / 'atom_map.json', mapping)
            model = args[0] if args else kwargs['molecular_system']
            inspection = inspect_native(model)
            save(directory / 'view_inspection.json', inspection)
            export_views.append(dict(model=model, inspection=inspection))
            record = dict(
                index=observed['writer_exports'],
                ligand_id=current['ligand_id'],
                run_id=None if active_item is None else active_item['run_id'],
                payload_sha256=hashlib.sha256(payload.encode()).hexdigest(),
                relative_directory=str(directory.relative_to(output)),
                atom_map=mapping,
            )
            writer_records.append(detached_record(record))
            save(directory / 'record.json', record)
        return value

    patch(msm, 'convert', convert)
    patch(msm, 'get', watch(msm.get, 'public_get_alias'))
    original_merge = msm.build.merge_nonpolar_hydrogens

    @wraps(original_merge)
    def merge(*args, **kwargs):
        assert observed['internal_merge'] < 2
        observed['internal_merge'] += 1
        return original_merge(*args, **kwargs)

    patch(msm.build, 'merge_nonpolar_hydrogens', merge)
    original_reader = msm.form.string_pdbqt_text.read_poses

    @wraps(original_reader)
    def read_poses(*args, **kwargs):
        assert observed['known_source_readers'] < 2
        observed['known_source_readers'] += 1
        value = original_reader(*args, **kwargs)
        directory = output / 'items' / active_item['run_id']
        report = detached_record(value['report'])
        save(directory / 'known_source_read_report.json', report)
        inspection = inspect_native(value['molecular_system'])
        save(directory / 'known_source_read_inspection.json', inspection)
        reader_records.append(dict(run_id=active_item['run_id'], report=report))
        return value

    patch(msm.form.string_pdbqt_text, 'read_poses', read_poses)
    for owner, name in (
        (dmt, 'score'),
        (dmt, 'compare_scores'),
        (dmt, 'prepare_receptor'),
        (dmt.VinaBackend, 'score'),
        (msm.build, 'add_missing_hydrogens'),
    ):
        patch(owner, name, reject(name))


def reproduce_candidate(candidate):
    directory = output / candidate['ligand_id']
    directory.mkdir()
    native, report = direct(
        'sdf_native',
        'sdf_native',
        msm.convert,
        str(inputs / Path(candidate['path']).name),
        to_form='molsysmt.MolSys',
        stereo_engine='rdkit',
        discard_properties=True,
        return_report=True,
    )
    save(directory / 'source_conversion_report.json', asdict(report))
    before = inspect_native(native)
    save(directory / 'source_native_inspection.json', before)
    live_sources.append((native, before))
    compare_source_observation(
        before, source_record(candidate['ligand_id'], 'source_native_inspection.json')
    )
    rdkit_model, report = direct(
        'native_to_rdkit',
        'native_to_rdkit',
        msm.convert,
        native,
        to_form='rdkit.Mol',
        return_report=True,
    )
    save(directory / 'native_to_rdkit_report.json', asdict(report))
    authority, report = direct(
        'rdkit_to_native',
        'rdkit_to_native',
        msm.convert,
        rdkit_model,
        to_form='molsysmt.MolSys',
        return_report=True,
    )
    save(directory / 'rdkit_to_native_report.json', asdict(report))
    authority_before = inspect_native(authority)
    save(directory / 'authority_native_inspection.json', authority_before)
    live_sources.append((authority, authority_before))
    compare_source_observation(
        authority_before,
        source_record(candidate['ligand_id'], 'authority_native_inspection.json'),
    )
    q_result = direct(
        'charge_assignment',
        'charge_assignment',
        msm.build.assign_partial_charges,
        authority,
        method='gasteiger_marsili',
        structure_indices=0,
        chemical_state='structure',
        expected_total_charge=puw.quantity(0, 'elementary_charge'),
        return_report=True,
    )
    save(directory / 'charge_calculation_report.json', q_result['report'])
    charged = q_result['molecular_system']
    charged_before = inspect_native(charged)
    save(directory / 'charged39_inspection.json', charged_before)
    live_sources.append((charged, charged_before))
    compare_inspection(
        charged_before, prep_record(candidate['ligand_id'], 'charged39_inspection.json')
    )
    t_result = direct(
        'type_assignment',
        'type_assignment',
        msm.build.assign_autodock_atom_types,
        charged,
        typing_scheme='autodock4',
        method='chemical_environment',
        structure_indices=0,
        chemical_state='structure',
        return_report=True,
    )
    save(directory / 'type_calculation_report.json', t_result['report'])
    typed = t_result['molecular_system']
    typed_before = inspect_native(typed)
    save(directory / 'typed39_inspection.json', typed_before)
    live_sources.append((typed, typed_before))
    compare_inspection(
        typed_before, prep_record(candidate['ligand_id'], 'typed39_inspection.json')
    )
    prepared = direct(
        'prepare',
        'prepare',
        dmt.prepare_ligand,
        typed,
        selection='all',
        state_id=candidate['state_id'],
        active_torsion_bonds=[],
    )
    declaration = detached_record(prepared.to_dict())
    coordinates = detached_record(
        dict(
            value=puw.get_value(prepared.coordinates, to_unit='angstrom'),
            unit='angstrom',
        )
    )
    save(directory / 'prepared_declaration.json', declaration)
    save(directory / 'prepared_coordinates.json', coordinates)
    live_preparations.append((prepared, declaration, coordinates))
    prepared_before = inspect_native(prepared.source_molsys)
    save(directory / 'prepared_source_inspection.json', prepared_before)
    live_sources.append((prepared.source_molsys, prepared_before))
    compare_inspection(
        prepared_before,
        prep_record(candidate['ligand_id'], 'prepared_source_inspection.json'),
    )
    prior_coordinates = prep_record(candidate['ligand_id'], 'prepared_coordinates.json')
    assert coordinates['unit'] == prior_coordinates['unit'] == 'angstrom'
    np.testing.assert_allclose(
        coordinates['value'], prior_coordinates['value'], rtol=0, atol=1e-12
    )
    assert prepared.n_atoms == 25 and prepared.torsion_dof == 0
    assert prepared.state_id == candidate['state_id']
    assert prepared.pdbqt_atom_indices == list(range(25))
    assert prepared.metadata['torsion_policy'] == 'rigid_only'
    assert prepared.metadata['active_torsion_bonds'] == []
    payload = direct('to_pdbqt', 'to_pdbqt', prepared.to_pdbqt)
    (directory / 'prepared.pdbqt').write_text(payload)
    actual_sha = hashlib.sha256(payload.encode()).hexdigest()
    assert actual_sha == expected_pdbqt[candidate['ligand_id']]
    mapping = writer_records[-1]['atom_map']
    assert writer_records[-1]['run_id'] is None
    assert mapping['payload_sha256'] == actual_sha
    assert mapping['source_atom_ids'] == prepared_before['ids']
    assert mapping == prep_record(candidate['ligand_id'], 'export_map.json')
    save(
        directory / 'reproduction_gate.json',
        dict(
            status='PASS',
            ligand_id=candidate['ligand_id'],
            state_id=prepared.state_id,
            expected_pdbqt_sha256=expected_pdbqt[candidate['ligand_id']],
            actual_pdbqt_sha256=actual_sha,
            source_archive_sha256=archive_sha,
            preparation_archive_sha256=preparation_archive_sha,
            n_atoms=25,
            torsion_dof=0,
            chemistry_and_mechanics_observations_compared=True,
            inspection_json_restored=False,
            current_new_charge_and_type_assignments=True,
        ),
    )
    return prepared


def install_native_observers(vina_module):
    for name in (
        '__init__',
        'set_receptor',
        'set_ligand_from_string',
        'set_ligand_from_file',
        'compute_vina_maps',
        'dock',
        'energies',
        'poses',
        'info',
        'score',
        'optimize',
        'randomize',
        'set_weights',
    ):
        original = getattr(vina_module.Vina, name)
        signature = inspect.signature(original)

        def create_observer(function, signature, method):
            @wraps(function)
            def call(*args, **kwargs):
                bound = signature.bind(*args, **kwargs)
                supplied = dict(bound.arguments)
                supplied.pop('self', None)
                bound.apply_defaults()
                resolved = dict(bound.arguments)
                resolved.pop('self', None)
                counter = 'constructor' if method == '__init__' else method
                if method == 'poses':
                    counter = (
                        'poses_coordinates'
                        if resolved['coordinates_only']
                        else 'poses_text'
                    )
                if native_caps[counter] == 0:
                    return reject('vina.' + method)()
                assert active_item is not None
                assert native_counts[counter] < native_caps[counter], counter
                native_counts[counter] += 1
                number = native_counts[counter]
                instance = args[0]
                if counter == 'constructor':
                    native_instances[instance] = f'vina_{number}'
                instance_id = native_instances[instance]
                directory = output / 'items' / active_item['run_id'] / 'native'
                directory.mkdir(parents=True, exist_ok=True)
                stem = f'{counter}_{number}'
                save(
                    directory / (stem + '_call.json'),
                    dict(
                        method=method,
                        counter=counter,
                        instance_id=instance_id,
                        supplied=supplied,
                        resolved=resolved,
                    ),
                )
                if method == 'set_receptor':
                    path = Path(resolved['rigid_pdbqt_filename']).resolve()
                    assert path.is_relative_to(tmpdir) and path.is_file()
                    content = path.read_bytes()
                    (directory / (stem + '.pdbqt')).write_bytes(content)
                    receptor_paths.append(
                        dict(
                            run_id=active_item['run_id'],
                            path=str(path),
                            sha256=hashlib.sha256(content).hexdigest(),
                        )
                    )
                if method == 'set_ligand_from_string':
                    (directory / (stem + '.pdbqt')).write_text(resolved['pdbqt_string'])
                event('native_started', method=method, native_counter=counter)
                value = function(*args, **kwargs)
                if counter in ('energies', 'poses_coordinates'):
                    # Retain exact arrays before finite JSON checks, without pickled objects.
                    np.save(
                        directory / (stem + '.npy'),
                        np.asarray(value),
                        allow_pickle=False,
                    )
                    binary = directory / (stem + '.npy')
                    save(
                        directory / (stem + '_binary.json'),
                        dict(
                            filename=binary.name,
                            sha256=file_sha(binary),
                            bytes=binary.stat().st_size,
                            shape=list(np.asarray(value).shape),
                            dtype=str(np.asarray(value).dtype),
                            unit='kcal/mol' if counter == 'energies' else 'angstrom',
                        ),
                    )
                    record = dict(
                        value=np.asarray(value).tolist(),
                        shape=list(np.asarray(value).shape),
                        unit='kcal/mol' if counter == 'energies' else 'angstrom',
                    )
                    if counter == 'energies':
                        record['columns'] = [
                            'total',
                            'inter',
                            'intra',
                            'torsions',
                            'intra_best_pose',
                        ]
                elif counter == 'poses_text':
                    (directory / (stem + '.pdbqt')).write_text(value)
                    record = dict(
                        text_sha256=hashlib.sha256(value.encode()).hexdigest(),
                        text_bytes=len(value.encode()),
                    )
                else:
                    record = dict(value=value)
                save(directory / (stem + '_return.json'), record)
                native_records.append(
                    detached_record(
                        dict(
                            run_id=active_item['run_id'],
                            method=method,
                            counter=counter,
                            instance_id=instance_id,
                            number=number,
                            resolved=resolved,
                            returned=record,
                        )
                    )
                )
                event('native_returned', method=method, native_counter=counter)
                return value

            return call

        patch(vina_module.Vina, name, create_observer(original, signature, name))


def public_outcome_roundtrips():
    rows = []
    for ref in outcome_records:
        assert counts['public_outcome_restores'] < budgets['public_outcome_restores']
        counts['public_outcome_restores'] += 1
        saved = ref['record']
        restored = dmt.DockingOutcome.from_dict(saved)
        returned = detached_record(restored.to_dict())
        directory = output / 'items' / ref['run_id']
        save(directory / 'restored_outcome.json', returned)
        a, b = detached_record(saved), detached_record(returned)
        physical = []
        if a['result'] is not None:
            for index, (original, reread) in enumerate(
                zip(a['result']['poses'], b['result']['poses'])
            ):
                before, after = original.pop('coordinates'), reread.pop('coordinates')
                x = puw.get_value(
                    puw.quantity(before['value'], before['unit']), to_unit='angstrom'
                )
                y = puw.get_value(
                    puw.quantity(after['value'], after['unit']), to_unit='angstrom'
                )
                np.testing.assert_allclose(x, y, rtol=0, atol=1e-12)
                np.testing.assert_allclose(
                    y,
                    puw.get_value(
                        restored.result.poses[index].coordinates, to_unit='angstrom'
                    ),
                    rtol=0,
                    atol=1e-12,
                )
                physical.append(
                    dict(
                        pose_index=index,
                        pose_id=original['pose_id'],
                        original_unit=before['unit'],
                        restored_unit=after['unit'],
                        comparison_unit='angstrom',
                        absolute_tolerance=1e-12,
                    )
                )
        assert a == b
        assert detached_record(outcomes[ref['input_position']].to_dict()) == saved
        rows.append(
            dict(
                run_id=ref['run_id'],
                outcome_sha256=ref['sha256'],
                normalized_noncoordinate_json_equal=True,
                physical_coordinates=physical,
                scores_ids_ranks_states_and_histories_equal=True,
                source_nonmutated=True,
            )
        )
    save(
        output / 'public_outcome_roundtrip.json',
        dict(
            schema='dockingmt.e2_public_outcome_roundtrip@1',
            status='PASS',
            public_outcome_restores=counts['public_outcome_restores'],
            rows=rows,
            native_calls_added=0,
            molecular_operations_added=0,
        ),
    )


def build_study_report(policy, policy_sha, outcome_records):
    """Finite caller report; molecular receiving belongs to the linked byte gates."""
    source = json.loads(json.dumps(outcome_records, allow_nan=False))
    refs = [
        {k: row[k] for k in ('run_id', 'input_position', 'path', 'sha256')}
        for row in source
    ]
    records = {row['input_position']: row for row in source}
    assert len(records) == len(source) <= 3 and set(records) <= {0, 1, 2}
    columns = ['total', 'inter', 'intra', 'torsions', 'intra_best_pose']
    score_columns = dict(vina=0, inter=1, intra=2, torsion=3)
    ligands = {row['ligand_id']: row for row in policy['ligands']}
    shared_contexts = []
    items = []

    def check(condition, reason, detail):
        if not condition:
            raise ValueError(reason + ': ' + detail)

    def numeric(value):
        check(
            type(value) in (int, float) and math.isfinite(value),
            'missing_nonfinite_or_unknown_score_definition',
            'finite scalar',
        )
        return float(value)

    def physical(record, unit):
        return puw.get_value(
            puw.quantity(record['value'], record['unit']), to_unit=unit
        )

    def same_physical(record, expected, unit):
        np.testing.assert_allclose(physical(record, unit), expected, rtol=0, atol=1e-12)

    counts = dict(
        submitted_items=3,
        distinct_ligands=2,
        experimental_complexes=1,
        items_retrieved=len(source),
        attempted_items=len(source),
        terminal_outcomes=len(source),
        successful_results=0,
        expected_refusals=0,
        unexpected_failures=0,
        not_attempted_or_missing=0,
        successful_empty_results=0,
        returned_poses=0,
        selected_representatives=0,
        eligible_representatives=0,
        incomparable_representatives=0,
        native_invocations=None,
    )
    for expected in policy['run_sequence']:
        position = expected['input_position']
        ligand = ligands[expected['ligand_id']]
        item = dict(
            run_id=expected['run_id'],
            input_position=position,
            ligand_id=expected['ligand_id'],
            declared_state_id=ligand['declared_state_id'],
            api_status=None,
            study_disposition='unranked',
            reasons=[],
            selected_original_pose_id=None,
            selected_original_pose_index=None,
            selected_original_rank=None,
            native_energy_row_index=None,
            native_energy_row=None,
            native_energy_columns=None,
            native_energy_unit=None,
            selected_native_total=None,
            reported_reference_component=None,
            observed_context_and_input_bindings={},
        )
        items.append(item)
        row = records.get(position)
        if row is None:
            item['reasons'].append('not_attempted_or_missing_outcome')
            counts['not_attempted_or_missing'] += 1
            continue
        record = row['record']
        try:
            reason = 'source_state_input_or_capture_binding_mismatch'
            payload = (json.dumps(record, indent=2, allow_nan=False) + '\n').encode()
            check(
                hashlib.sha256(payload).hexdigest() == row['sha256'],
                reason,
                'outcome wire',
            )
            check(
                row['run_id'] == expected['run_id'] and record['index'] == position,
                reason,
                'item association',
            )
            item['api_status'] = record['status']
            if record['status'] == 'failure':
                error = record['error']
                if (
                    position == 1
                    and error['code'] == 'DMT-E003'
                    and error['type'].endswith('.CapabilityMismatchError')
                ):
                    check(
                        record['result'] is None
                        and record['problem_info']['constraints']
                        == expected['constraints'],
                        reason,
                        'refusal association',
                    )
                    item['study_disposition'] = 'expected_refusal'
                    item['reasons'].append('expected_capability_refusal')
                    counts['expected_refusals'] += 1
                else:
                    item['reasons'].append('unexpected_item_failure')
                    counts['unexpected_failures'] += 1
                continue
            check(
                record['status'] == 'success' and position != 1,
                reason,
                'scientific outcome',
            )
            counts['successful_results'] += 1
            result = record['result']
            poses = result['poses']
            counts['returned_poses'] += len(poses)
            if not poses:
                item['reasons'].append('successful_empty_result')
                counts['successful_empty_results'] += 1
                continue
            reason = 'missing_or_ambiguous_original_rank_1'
            ranks = [
                i
                for i, pose in enumerate(poses)
                if type(pose.get('rank')) is int and pose['rank'] == 1
            ]
            check(
                len(ranks) == 1 and ranks[0] == 0,
                reason,
                'unique original rank-1 position',
            )
            selected_index = ranks[0]
            selected = poses[selected_index]
            item.update(
                selected_original_pose_id=selected['pose_id'],
                selected_original_pose_index=selected_index,
                selected_original_rank=1,
            )
            history = result['provenance']['ranking_history']
            check(
                len(history) == 1
                and history[0]['origin'] == 'backend'
                and history[0]['evidence'] == 'recorded',
                reason,
                'original backend history',
            )
            check(
                history[0]['score_name'] == 'vina' and history[0]['ascending'] is True,
                reason,
                'backend total decision',
            )
            check(
                history[0]['output_order'] == list(range(len(poses))),
                reason,
                'original output order',
            )
            observations = [
                dict(
                    pose_id=p['pose_id'],
                    partner_state_id=p['partner_state_id'],
                    receptor_state_id=p['receptor_state_id'],
                    rank=p['rank'],
                    value=p['scores']['vina'],
                )
                for p in poses
            ]
            check(
                history[0]['input_order'] == observations,
                reason,
                'original observations',
            )
            check(
                result['provenance']['ranking_policy']
                == dict(score_name='vina', ascending=True),
                reason,
                'retained ranking policy',
            )
            reason = 'source_state_input_or_capture_binding_mismatch'
            problem = result['problem_info']
            provenance = result['provenance']
            check(
                record['problem_info'] == problem
                and record['protocol_info'] == result['protocol_info'],
                reason,
                'outcome/result copies',
            )
            check(
                problem['metadata']['run_id'] == expected['run_id']
                and problem['metadata']['ligand_id'] == ligand['ligand_id'],
                reason,
                'run identities',
            )
            check(
                problem['metadata']['partner_state_id'] == ligand['declared_state_id'],
                reason,
                'declared partner state',
            )
            check(
                problem['metadata']['receptor_state_id']
                == policy['common_receptor']['declared_state_id'],
                reason,
                'declared receptor state',
            )
            check(
                problem['constraints'] == expected['constraints']
                and problem['search_guidance'] == expected['search_guidance'],
                reason,
                'intent',
            )
            captured = {}
            for role, digest in (
                ('receptor', policy['common_receptor']['sha256']),
                ('partner', ligand['prepared_pdbqt_sha256']),
            ):
                artifact = provenance['backend_artifacts'][role]
                raw = base64.b64decode(artifact['content_base64'], validate=True)
                check(
                    artifact['format'] == 'pdbqt'
                    and hashlib.sha256(raw).hexdigest() == artifact['sha256'] == digest,
                    reason,
                    'captured ' + role,
                )
                captured[role] = dict(sha256=digest, bytes=len(raw))
            prepared = problem['partner']['value']
            meta = prepared['metadata']
            check(
                prepared['state_id'] == ligand['declared_state_id']
                and prepared['n_atoms'] == 25,
                reason,
                'own prepared declaration',
            )
            check(
                prepared['torsion_dof'] == 0
                and prepared['pdbqt_atom_indices'] == list(range(25)),
                reason,
                'rigid prepared axis',
            )
            item['observed_context_and_input_bindings'] = dict(
                captures=captured,
                registered_source_sha256=ligand['source_sha256'],
                registered_preparation_archive_sha256=policy['input_manifests'][
                    'received_preparation_archive'
                ]['sha256'],
                prepared_state_id=prepared['state_id'],
            )
            reason = 'missing_or_inconsistent_five_column_native_evidence'
            native = provenance['backend_output']
            check(
                native['format'] == 'vina_python_docking@1'
                and native['energy_columns'] == columns
                and native['energy_unit'] == 'kcal/mol',
                reason,
                'five canonical native columns',
            )
            energies = native['energies']
            check(
                len(energies) == len(poses) <= 3, reason, 'native row/pose population'
            )
            check(
                all(
                    len(values) == 5 and all(math.isfinite(numeric(x)) for x in values)
                    for values in energies
                ),
                reason,
                'native rows',
            )
            for index, pose in enumerate(poses):
                check(
                    pose['rank'] == index + 1
                    and pose['partner_state_id'] == ligand['declared_state_id']
                    and pose['receptor_state_id']
                    == policy['common_receptor']['declared_state_id'],
                    reason,
                    'pose row identities',
                )
                check(
                    set(pose['scores']) == set(score_columns),
                    reason,
                    'four original public scores',
                )
                for name, column in score_columns.items():
                    check(
                        numeric(pose['scores'][name])
                        == numeric(energies[index][column]),
                        reason,
                        'own native row ' + name,
                    )
            item.update(
                native_energy_row_index=selected_index,
                native_energy_row=list(energies[selected_index]),
                native_energy_columns=list(columns),
                native_energy_unit='kcal/mol',
                selected_native_total=numeric(energies[selected_index][0]),
                reported_reference_component=numeric(energies[selected_index][4]),
            )
            reason = 'incompatible_method_weights_protocol_receptor_domain_grid_or_preparation_context'
            p = policy['protocol']
            defaults = policy['observed_native_defaults']
            check(
                provenance['backend'] == 'vina'
                and provenance['backend_version'] == '1.2.7'
                and provenance['seed'] == p['seed'],
                reason,
                'backend and seed',
            )
            info = native['engine_info']
            check(
                info['scoring_function'] == 'vina'
                and info['seed'] == p['seed']
                and info['no_refine'] is False,
                reason,
                'native method',
            )
            weights = [numeric(x) for x in info['weights']]
            check(
                weights == [numeric(x) for x in defaults['weights']],
                reason,
                'actual default weights',
            )
            box = {
                key: policy['common_domain'][key] for key in ('center', 'size', 'unit')
            }
            check(
                info['box_center'] == box['center']
                and info['box_size'] == box['size']
                and info['box_spacing'] == 0.375,
                reason,
                'actual grid box',
            )
            check(
                type(info['box_even_elements']) is bool, reason, 'actual voxel parity'
            )
            check(
                {key: provenance['backend_box'][key] for key in box} == box
                and provenance['backend_box']['rounding_decimals'] == 6,
                reason,
                'backend box projection',
            )
            calls = native['resolved_calls']
            check(
                calls['compute_vina_maps']
                == dict(
                    center=box['center'],
                    box_size=box['size'],
                    spacing=0.375,
                    force_even_voxels=False,
                ),
                reason,
                'resolved grid call',
            )
            check(
                calls['dock']
                == dict(exhaustiveness=1, n_poses=3, min_rmsd=1.0, max_evals=0),
                reason,
                'resolved search call',
            )
            check(
                calls['energies'] == dict(n_poses=3, energy_range=3.0),
                reason,
                'resolved energy retrieval',
            )
            check(
                problem['search_domain']['type'] == 'BoxRegion', reason, 'domain type'
            )
            same_physical(problem['search_domain']['center'], box['center'], 'angstrom')
            same_physical(problem['search_domain']['lengths'], box['size'], 'angstrom')
            check(
                provenance['search_domain'] == problem['search_domain'],
                reason,
                'domain record copies',
            )
            for declaration in (result['protocol_info'], provenance['protocol']):
                check(
                    declaration['protocol_type'] == 'VinaProtocol',
                    reason,
                    'protocol type',
                )
                actual = declaration['parameters']
                keys = (
                    'scoring',
                    'cpu',
                    'seed',
                    'exhaustiveness',
                    'n_poses',
                    'capture_backend_inputs',
                    'collect_timings',
                    'allow_provisional_preparation',
                )
                check(
                    all(actual[key] == p[key] for key in keys)
                    and actual['active_torsion_bonds'] == [],
                    reason,
                    'declared protocol',
                )
                same_physical(actual['energy_range'], 3.0, 'kcal/mol')
            check(
                meta['torsion_policy'] == 'rigid_only'
                and meta['active_torsion_bonds'] == []
                and meta['hydrogen_policy'] == 'retain_polar_merge_nonpolar',
                reason,
                'declared rigid/H policy',
            )
            q, typing = meta['partial_charge_assignment'], meta['atom_type_assignment']
            check(
                q['method'] == 'gasteiger_marsili' and q['coverage'] == 'complete',
                reason,
                'named charge policy',
            )
            check(
                typing['typing_scheme'] == 'autodock4'
                and typing['method'] == 'chemical_environment'
                and typing['rule_version'] == 'chemical_environment@1'
                and typing['coverage'] == 'complete',
                reason,
                'named type policy',
            )
            preparation = provenance['preparation']
            check(
                preparation['receptor']['assessment'] == 'unassessed'
                and preparation['partner']['mode'] == 'provided',
                reason,
                'provided preparations',
            )
            check(
                preparation['partner']['metadata'] == meta
                and preparation['partner']['provisional_reasons'] == [],
                reason,
                'preparation copies and policy',
            )
            context = dict(
                backend='vina',
                version='1.2.7',
                weights=weights,
                no_refine=info['no_refine'],
                box_even_elements=info['box_even_elements'],
                box=provenance['backend_box'],
                resolved_calls=calls,
                receptor_sha256=captured['receptor']['sha256'],
                receptor_state_id=policy['common_receptor']['declared_state_id'],
                protocol=provenance['protocol'],
                hydrogen_policy=meta['hydrogen_policy'],
                torsion_policy=meta['torsion_policy'],
                charge_policy={
                    key: q[key]
                    for key in (
                        'method',
                        'engine',
                        'software',
                        'parameters',
                        'charge_unit',
                        'expected_total_charge',
                    )
                },
                type_policy={
                    key: typing[key]
                    for key in (
                        'typing_scheme',
                        'method',
                        'rule_version',
                        'software',
                        'parameters',
                        'hydrogen_policy',
                    )
                },
                preparation_assessment={
                    role: preparation[role]['assessment']
                    for role in ('receptor', 'partner')
                },
            )
            item['observed_context_and_input_bindings']['shared_context'] = context
            reason = 'missing_nonfinite_or_unknown_score_definition'
            components = dict(
                vina='total docking score',
                inter='intermolecular term',
                intra='intramolecular term',
                torsion='torsional term',
            )
            for pose in poses:
                definitions = pose['metadata']['score_definitions']
                for name, component in components.items():
                    definition = definitions[name]
                    check(
                        definition['schema_version'] == '1.0'
                        and definition['method'] == 'AutoDock Vina/vina'
                        and definition['method_version'] == '1.2.7'
                        and definition['component'] == component
                        and definition['kind'] == 'empirical',
                        reason,
                        'native descriptor ' + name,
                    )
                    same_physical(
                        dict(value=1.0, unit=definition['unit']), 1.0, 'kcal/mol'
                    )
                    expected_context = dict(
                        stage='docking',
                        receptor_sha256=captured['receptor']['sha256'],
                        partner_sha256=captured['partner']['sha256'],
                        backend_box=box,
                        weights=weights,
                        grid_spacing=dict(value=info['box_spacing'], unit='angstrom'),
                        preparation_assessment=context['preparation_assessment'],
                    )
                    check(
                        definition['context'] == expected_context,
                        reason,
                        'own descriptor context',
                    )
                check(
                    definitions['vina']['preferred_direction'] == 'lower',
                    reason,
                    'native total direction',
                )
            check(
                history[0]['score_definition']
                == selected['metadata']['score_definitions']['vina'],
                reason,
                'ranking descriptor',
            )
            item['study_disposition'] = 'eligible'
            shared_contexts.append((item, context))
        except (KeyError, TypeError, ValueError, IndexError, AssertionError) as error:
            item['study_disposition'] = 'incomparable'
            item['reasons'].append(reason)
            item['observed_context_and_input_bindings']['diagnostic'] = str(error)
    if len(shared_contexts) == 2 and shared_contexts[0][1] != shared_contexts[1][1]:
        for item, _ in shared_contexts:
            item['study_disposition'] = 'incomparable'
            item['reasons'].append(
                'incompatible_method_weights_protocol_receptor_domain_grid_or_preparation_context'
            )
    eligible = sorted(
        (row for row in items if row['study_disposition'] == 'eligible'),
        key=lambda row: (row['selected_native_total'], row['input_position']),
    )
    groups = []
    for item in eligible:
        if (
            not groups
            or groups[-1]['selected_native_total'] != item['selected_native_total']
        ):
            groups.append(
                dict(
                    selected_native_total=item['selected_native_total'],
                    unit='kcal/mol',
                    members=[],
                )
            )
        groups[-1]['members'].append(
            {
                key: item[key]
                for key in (
                    'run_id',
                    'input_position',
                    'ligand_id',
                    'selected_original_pose_id',
                    'selected_original_pose_index',
                    'selected_original_rank',
                    'native_energy_row_index',
                )
            }
        )
    counts['selected_representatives'] = sum(
        row['selected_original_pose_id'] is not None for row in items
    )
    counts['eligible_representatives'] = len(eligible)
    counts['incomparable_representatives'] = sum(
        row['selected_original_pose_id'] is not None
        and row['study_disposition'] != 'eligible'
        for row in items
    )
    if counts['unexpected_failures'] == counts['not_attempted_or_missing'] == 0:
        counts['native_invocations'] = counts['successful_results']
    report = dict(
        schema='dockingmt.e2_native_study_report@1',
        study_id=policy['study_id'],
        policy_id=policy['policy_id'],
        policy_sha256=policy_sha,
        outcome_records=refs,
        items=items,
        order_groups=groups,
        counts=counts,
        complete_two_ligand_order=len(eligible) == 2
        and {row['ligand_id'] for row in eligible} == {'p59', 'p69'},
        limitations=[
            policy['scope'],
            policy['study_comparison']['interpretation'],
            'Pure case-specific record consistency; no producer authentication. Molecular source/preparation receiving belongs to the two exact pre-native byte gates linked to the source/preparation archives.',
            'Each native search keeps its own fifth-column reference; no common reference, recombination, affinity, stereoselectivity or uncertainty claim.',
            'Canonical raw native kcal/mol totals determine exact ties; public physical conversion checks the recorded domain and protocol units.',
            'Native invocation counts inferred from successful records and the pinned pre-native refusal; actual observer counts remain in the separate receipt.',
            'No original scores, ranks, order or histories are changed; E1 compare_scores/rank_by and whole-E2 acceptance remain outside this report.',
        ],
    )
    return json.loads(json.dumps(report, allow_nan=False))

In [ ]:
failure = None
outcome = 'FAIL'
try:
    installed_check('before-e2-native-collection')
    integrity['before'] = True
    source_identity()
    assert policy['schema'] == 'dockingmt.e2_native_study_policy@1'
    assert policy['owner'] == 'uibcdf/dockingmt#81'
    with puw.context(
        standard_units=[
            'pm',
            'fs',
            'kelvin',
            'mole',
            'dalton',
            'coulomb',
            'kJ/mol',
            'kJ/(mol*pm)',
            'kJ/(mol*pm**2)',
            'radians',
        ]
    ):
        active_policy = detached_record(puw.configure.report())
        save(output / 'active_unit_policy.json', active_policy)
        install_monitoring()
        for position, candidate in enumerate(expected_candidates):
            current = ledger[position]
            current['status'] = 'attempted'
            persist_ledger()
            prepared_by_id[candidate['ligand_id']] = reproduce_candidate(candidate)
            records.append(
                dict(
                    ligand_id=candidate['ligand_id'],
                    pdbqt_sha256=expected_pdbqt[candidate['ligand_id']],
                )
            )
            current['status'] = 'admitted'
            persist_ledger()
        stage = 'both_pre_native_byte_gates'
        assert len(prepared_by_id) == 2
        assert observed['internal_merge'] == observed['writer_exports'] == 2
        assert not any(name.split('.')[0] == 'vina' for name in sys.modules)
        assert not forbidden_attempts
        save(
            output / 'pre_native_gates.json',
            dict(
                status='PASS',
                prepared=records,
                source_archive_sha256=archive_sha,
                preparation_archive_sha256=preparation_archive_sha,
                direct_counts=counts,
                observed=observed,
                vina_imported=False,
                both_preparations_checked_before_first_native_import=True,
            ),
        )
        sys.meta_path.remove(import_guard)
        import vina

        assert vina.__version__ == policy['protocol']['backend_version'] == '1.2.7'
        for name in ('vina', 'vina.vina', 'vina._vina_wrapper', 'vina._vina'):
            module = sys.modules.get(name)
            if module is not None and getattr(module, '__file__', None):
                path = Path(module.__file__).resolve()
                identities[name] = dict(path=str(path), sha256=file_sha(path))
        save(output / 'native_module_identities.json', identities)
        install_native_observers(vina)
        p = policy['protocol']
        assert (
            p['scoring'],
            p['cpu'],
            p['seed'],
            p['exhaustiveness'],
            p['n_poses'],
        ) == ('vina', 1, 42, 1, 3)
        assert p['energy_range'] == dict(value=3.0, unit='kcal/mol')
        assert p['capture_backend_inputs'] and p['collect_timings']
        assert p['allow_provisional_preparation'] is False
        protocol = dmt.VinaProtocol(
            scoring='vina',
            cpu=1,
            seed=42,
            exhaustiveness=1,
            n_poses=3,
            energy_range=puw.quantity(3, 'kcal/mol'),
            capture_backend_inputs=True,
            collect_timings=True,
            allow_provisional_preparation=False,
            active_torsion_bonds=[],
        )
        domain_spec = policy['common_domain']
        assert domain_spec['center'] == [-15.0, 15.0, 129.0]
        assert (
            domain_spec['size'] == [30.0, 24.0, 24.0]
            and domain_spec['unit'] == 'angstrom'
        )
        domain = dmt.BoxRegion(
            center=puw.quantity(domain_spec['center'], 'angstrom'),
            size=puw.quantity(domain_spec['size'], 'angstrom'),
        )
        receptor = str(inputs / Path(policy['common_receptor']['path']).name)
        assert file_sha(Path(receptor)) == policy['common_receptor']['sha256']
        assert policy['run_sequence'][1]['constraints'] == [
            {'kind': 'e2_unsupported_intent_control'}
        ]
        problems = []
        for spec in policy['run_sequence']:
            problems.append(
                dmt.DockingProblem(
                    receptor=receptor,
                    partner=prepared_by_id[spec['ligand_id']],
                    search_domain=domain,
                    constraints=spec['constraints'],
                    search_guidance=spec['search_guidance'],
                    metadata=dict(
                        study_id=policy['study_id'],
                        run_id=spec['run_id'],
                        ligand_id=spec['ligand_id'],
                        partner_state_id=prepared_by_id[spec['ligand_id']].state_id,
                        receptor_state_id=policy['common_receptor'][
                            'declared_state_id'
                        ],
                    ),
                )
            )
        problem_records = [detached_record(problem.to_dict()) for problem in problems]
        protocol_record = detached_record(protocol.to_dict())
        save(output / 'problem_declarations.json', problem_records)
        save(output / 'protocol_declaration.json', protocol_record)
        (output / 'items').mkdir()
        stage = 'dock_many'
        counts['dock_many'] += 1
        iterator = dmt.dock_many(
            problems, protocol=protocol, backend='vina', on_error='record'
        )
        for position, spec in enumerate(item_specs):
            current = item_ledger[position]
            active_item = spec
            current['status'] = 'attempted'
            current['native_counts_before'] = native_counts.copy()
            current['writer_exports_before'] = observed['writer_exports']
            current['caller_policy_before'] = detached_record(puw.configure.report())
            directory = output / 'items' / spec['run_id']
            directory.mkdir()
            persist_ledger()
            event('item_started', run_id=spec['run_id'])
            # Ordinary failures and empty results do not abort the three-item selection.
            # Source, MemoryError and process-control exceptions still propagate.
            returned = next(iterator)
            record = detached_record(returned.to_dict())
            save(directory / 'outcome.json', record)
            ref = dict(
                run_id=spec['run_id'],
                input_position=position,
                path=str((directory / 'outcome.json').relative_to(output)),
                sha256=file_sha(directory / 'outcome.json'),
                record=record,
            )
            outcomes.append(returned)
            outcome_records.append(ref)
            if record['result'] is not None:
                save(directory / 'original_result.json', record['result'])
            current['status'] = 'returned'
            current['api_status'] = record['status']
            current['native_counts_after'] = native_counts.copy()
            current['writer_exports_after'] = observed['writer_exports']
            current['caller_policy_after'] = detached_record(puw.configure.report())
            paths = [
                dict(**row, exists_after_return=Path(row['path']).exists())
                for row in receptor_paths
                if row['run_id'] == spec['run_id']
            ]
            save(directory / 'receptor_temporary_cleanup.json', dict(paths=paths))
            current['receptor_cleanup_checked'] = paths
            persist_ledger()
            event('item_returned', run_id=spec['run_id'], api_status=record['status'])
        active_item = None
        current = None
        stage = 'saved_collection_qualification'
        save(output / 'native_returns.json', native_records)
        save(output / 'known_source_read_returns.json', reader_records)
        save(output / 'writer_returns.json', writer_records)
        save(
            output / 'outcome_records.json',
            [
                {k: v for k, v in row.items() if k != 'record'}
                for row in outcome_records
            ],
        )
        public_outcome_roundtrips()
        study_inputs_before = detached_record(outcome_records)
        report = build_study_report(policy, policy_sha, outcome_records)
        save(output / 'study_report.json', report)
        assert detached_record(outcome_records) == study_inputs_before
        assert detached_record(policy) == policy_before
        for ref in outcome_records:
            assert file_sha(output / ref['path']) == ref['sha256']
        assert len(outcomes) == 3 and [
            r['record']['index'] for r in outcome_records
        ] == [0, 1, 2]
        assert [r['record']['status'] for r in outcome_records] == [
            'success',
            'failure',
            'success',
        ]
        error = outcome_records[1]['record']['error']
        assert (
            error['type'].endswith('.CapabilityMismatchError')
            and error['code'] == 'DMT-E003'
        )
        assert (
            item_ledger[1]['native_counts_before']
            == item_ledger[1]['native_counts_after']
        )
        assert (
            item_ledger[1]['writer_exports_before']
            == item_ledger[1]['writer_exports_after']
        )
        for row in item_ledger:
            assert (
                row['caller_policy_before']
                == row['caller_policy_after']
                == active_policy
            )
            assert all(
                not p['exists_after_return'] for p in row['receptor_cleanup_checked']
            )
        successful_nonempty = all(
            r['record']['result']['poses']
            for r in (outcome_records[0], outcome_records[2])
        )
        for ref in (outcome_records[0], outcome_records[2]):
            counts['audit_result'] += 1
            audit = dmt.audit_result(ref['record']['result'])
            save(output / 'items' / ref['run_id'] / 'audit.json', audit)
            if ref['record']['result']['poses']:
                assert audit['status'] == 'consistent'
        assert counts == budgets
        assert all(native_counts[key] <= value for key, value in native_caps.items())
        if successful_nonempty:
            assert native_counts == native_caps
            assert (
                observed['writer_exports'] == 4
                and observed['known_source_readers'] == 2
            )
            assert report['complete_two_ligand_order']
        for row in native_records:
            resolved = row['resolved']
            if row['counter'] == 'constructor':
                assert resolved == dict(
                    sf_name='vina', cpu=1, seed=42, no_refine=False, verbosity=0
                )
            elif row['counter'] == 'compute_vina_maps':
                assert resolved == dict(
                    center=[-15.0, 15.0, 129.0],
                    box_size=[30.0, 24.0, 24.0],
                    spacing=0.375,
                    force_even_voxels=False,
                )
            elif row['counter'] == 'dock':
                assert resolved == dict(
                    exhaustiveness=1, n_poses=3, min_rmsd=1.0, max_evals=0
                )
            elif row['counter'] == 'info':
                info = row['returned']['value']
                assert [float(v) for v in info['weights']] == [
                    float(v) for v in policy['observed_native_defaults']['weights']
                ]
                assert info['no_refine'] is False and info['scoring_function'] == 'vina'
                assert info['seed'] == 42 and info['box_spacing'] == 0.375
            elif row['counter'] == 'energies':
                assert resolved == dict(n_poses=3, energy_range=3.0)
        for ref in (outcome_records[0], outcome_records[2]):
            result = ref['record']['result']
            if not result['poses']:
                continue
            own = [r for r in native_records if r['run_id'] == ref['run_id']]
            energies = next(r['returned'] for r in own if r['counter'] == 'energies')
            coordinates = next(
                r['returned'] for r in own if r['counter'] == 'poses_coordinates'
            )
            text = next(r['returned'] for r in own if r['counter'] == 'poses_text')
            backend_output = result['provenance']['backend_output']
            assert backend_output['energies'] == energies['value']
            assert backend_output['energy_columns'] == energies['columns']
            assert (
                backend_output['pose_reading']['method']
                == 'molsysmt.form.string_pdbqt_text.read_poses'
            )
            assert (
                backend_output['pose_reading']['report']['input_payload_sha256']
                == text['text_sha256']
            )
            assert (
                len(result['poses']) == energies['shape'][0] == coordinates['shape'][0]
            )
            assert 1 <= len(result['poses']) <= 3 and coordinates['shape'][1:] == [
                25,
                3,
            ]
            ligand_id = item_specs[ref['input_position']]['ligand_id']
            prepared = prepared_by_id[ligand_id]
            for index, pose in enumerate(result['poses']):
                physical = puw.get_value(
                    puw.quantity(
                        pose['coordinates']['value'], pose['coordinates']['unit']
                    ),
                    to_unit='angstrom',
                )
                np.testing.assert_allclose(
                    physical, coordinates['value'][index], rtol=0, atol=1e-12
                )
                assert pose['metadata']['pose_atom_order'] == 'verified_pdbqt_order'
                assert pose['metadata']['prepared_atom_indices'] == list(range(25))
                keys = pose['metadata']['source_atom_keys']
                assert (
                    len(keys) == 25
                    and [key['atom_name'] for key in keys] == prepared.atom_names
                )
        for model, before in live_sources:
            assert inspect_native(model) == before
        for item in export_views:
            assert inspect_native(item['model']) == item['inspection']
        for model, declaration, coordinates in live_preparations:
            assert detached_record(model.to_dict()) == declaration
            np.testing.assert_array_equal(
                puw.get_value(model.coordinates, to_unit=coordinates['unit']),
                coordinates['value'],
            )
        assert [
            detached_record(problem.to_dict()) for problem in problems
        ] == problem_records
        assert detached_record(protocol.to_dict()) == protocol_record
        for ref, live in zip(outcome_records, outcomes):
            assert detached_record(live.to_dict()) == ref['record']
        for path, expected in {**stored_inputs, **input_copies}.items():
            assert file_sha(Path(path)) == expected
        assert detached_record(puw.configure.report()) == active_policy
        assert not forbidden_attempts
        assert detached_record(policy) == policy_before
        save(
            output / 'nonmutation.json',
            dict(
                status='PASS',
                native_sources_checked=len(live_sources),
                exported_views_checked=len(export_views),
                prepared_declarations_and_physical_coordinates_checked=len(
                    live_preparations
                ),
                problems_checked=3,
                protocols_checked=1,
                policy_dict_preserved=True,
                original_outcomes_checked=3,
                saved_input_files_checked=len(stored_inputs) + len(input_copies),
                caller_policy_preserved_inside_context=True,
                output_ranks_and_histories_mutated=False,
            ),
        )
        outcome = 'PASS' if successful_nonempty else 'INCOMPLETE'
except BaseException as error:
    failure = error
    if current is not None:
        current['status'] = 'failed'
        current['failed_stage'] = stage
    persist_ledger()
    save(
        output / 'failure.json',
        dict(
            stage=stage,
            error_type=type(error).__name__,
            reason=str(error),
            code=getattr(error, 'code', None),
            direct_counts=counts,
            native_counts=native_counts,
            observed=observed,
            retry_permitted=False,
        ),
    )
finally:
    for owner, name, original in reversed(patches):
        setattr(owner, name, original)
    if import_guard in sys.meta_path:
        sys.meta_path.remove(import_guard)
    try:
        assert detached_record(puw.configure.report()) == caller_policy
        installed_check('after-e2-native-collection')
        integrity['after'] = True
    except BaseException as error:
        save(
            output / 'integrity_failure.json',
            dict(error_type=type(error).__name__, reason=str(error)),
        )
        if failure is None:
            failure = error
        else:
            failure.add_note('Final integrity check failed: ' + str(error))
    if failure is not None:
        outcome = 'FAIL'
    persist_ledger()
    for filename, value in (
        ('native_returns.json', native_records),
        ('known_source_read_returns.json', reader_records),
        ('writer_returns.json', writer_records),
        (
            'outcome_records.json',
            [
                {k: v for k, v in row.items() if k != 'record'}
                for row in outcome_records
            ],
        ),
    ):
        if not (output / filename).exists():
            save(output / filename, value)
    save(
        output / 'receipt.json',
        dict(
            schema='dockingmt.e2_ligand_collection_receiving@1',
            owner='uibcdf/dockingmt#81',
            outcome=outcome,
            stage=stage,
            candidate_manifest_sha256=manifest_sha,
            source_archive_sha256=archive_sha,
            preparation_archive_sha256=preparation_archive_sha,
            study_policy_sha256=policy_sha,
            direct_operation_budgets=budgets,
            direct_operation_counts=counts,
            native_call_caps=native_caps,
            native_call_counts=native_counts,
            direct_count_scope='Explicit orchestration only; provider-internal conversion and SDF stereo interpretation are separate observations',
            observed_public_alias_calls_and_snapshots=observed,
            input_gate_exports=counts['to_pdbqt'],
            backend_staging_exports=sum(
                row['run_id'] is not None for row in writer_records
            ),
            observed_count_scope='Public aliases and snapshots; not every internal provider dispatch',
            source_input_sha256=stored_inputs,
            raw_input_copy_sha256=input_copies,
            source_license=candidate_manifest['source_license'],
            module_identities=identities,
            installed_integrity_checks=integrity,
            caller_unit_policy=caller_policy,
            caller_policy_restored=detached_record(puw.configure.report())
            == caller_policy,
            active_caller_units='pm/fs/coulomb',
            candidate_ligands=2,
            declared_items=3,
            attempted_items=sum(
                row['status'] != 'not_attempted' for row in item_ledger
            ),
            returned_outcomes=len(outcomes),
            ledger=dict(preparations=ledger, items=item_ledger),
            forbidden_attempts=forbidden_attempts,
            new_charge_assignments=counts['charge_assignment'],
            new_type_assignments=counts['type_assignment'],
            new_preparations=counts['prepare'],
            new_hydrogen_calls=0,
            native_searches=native_counts['dock'],
            native_energies_retrievals=native_counts['energies'],
            fixed_pose_score_calls=native_counts['score'],
            native_internal_energy_calculations='Not individually counted; native search calculations remain inside each declared dock',
            public_outcome_restores=counts['public_outcome_restores'],
            retries=0,
            retry_permitted=False,
            vina_imported=any(name.split('.')[0] == 'vina' for name in sys.modules),
            inspection_json_is_restoration_input=False,
            new_runtime_api=False,
            interpretation='Case-specific empirical native docking scores; no affinity/stereoselectivity/convergence/screening-effectiveness claim',
            receptor_preparation_assessment='external/unassessed',
            whole_e2_received=False,
        ),
    )
if failure is not None:
    raise failure